# Notebook 08: Vision to Decision

## Before You Start
> **If anything behaves unexpectedly, restart the kernel first: Kernel menu → Restart Kernel and Clear All Outputs. Then run the cells from the top.**

## ADAS Connection
In the last notebook you learned how to talk to an AI model and get driving decisions. In this notebook you connect that model to the robot's **actual camera feed** -- the robot will see something, describe it to the AI, and act on what the AI says.

This is called **sensor fusion with AI reasoning** -- combining raw sensor data (camera) with AI decision making. It is the core architecture of modern autonomous vehicles.

Real ADAS systems like Tesla's FSD do exactly this thousands of times per second:
1. Camera captures a frame
2. Computer vision extracts what is in the frame
3. AI model decides what to do
4. Motors execute the decision

Today you will build a slower but real version of this pipeline.

---

## How It Works
The pipeline has four stages:

```
CAMERA → DETECT COLOR → BUILD OBSERVATION → AI DECIDES → MOTORS ACT
```

The key new piece is **BUILD OBSERVATION** -- translating raw pixel data into a natural language description the AI can reason about. This translation layer is called a **perception summary**.

---

## The Code
Run this cell to set up camera, motors, and AI connection.

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
import threading
import time
import sys
from IPython.display import display

sys.path.insert(0, '/home/pi/lab')
import motors
import ai_driver

def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

# Open camera
try:
    cap.release()
except:
    pass

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
cap.set(cv2.CAP_PROP_BRIGHTNESS, 40)
cap.set(cv2.CAP_PROP_CONTRAST,   40)

# Color ranges
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'blue':   ([100, 43,  46],  [124, 255, 255]),
    'yellow': ([26,  43,  46],  [34,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

# Setup motors
motors.setup()

# Warm up camera
print('Warming up camera...')
for _ in range(20):
    cap.read()
    time.sleep(0.05)

# Warm up AI model
print('Warming up AI model...')
ai_driver.ask('hello')

print('Camera ready!')
print('Motors ready!')
print('AI model ready!')
print()
print('>>> Make sure the robot is on the floor with space to move.')

---

## YOUR TURN -- Tweak Zone 1: Build the Observation

This is the perception summary -- the translation from pixels to language. The function below looks at the camera frame and builds a text description of what it sees.

Change `TARGET_COLOR` and `MIN_RADIUS` then run the cell. Hold your colored object in front of the camera and see what observation gets built.

> **Think like an engineer:** The quality of the AI decision depends entirely on the quality of the observation. What information would you add to make the observation more useful?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
TARGET_COLOR = 'red'    # your team color
MIN_RADIUS   = 20       # minimum blob size
# ═══════════════════════════════════════

color_lower = np.array(COLOR_RANGES[TARGET_COLOR][0])
color_upper = np.array(COLOR_RANGES[TARGET_COLOR][1])
FRAME_CENTER = 320

def build_observation(frame):
    """Translate camera frame into a text observation for the AI."""
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, color_lower, color_upper)
    mask = cv2.erode(mask,  None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    mask = cv2.GaussianBlur(mask, (3,3), 0)
    cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    
    if len(cnts) > 0:
        cnt = max(cnts, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(cnt)
        
        if radius > MIN_RADIUS:
            # determine position
            if cx < FRAME_CENTER - 50:
                position = "left side"
            elif cx > FRAME_CENTER + 50:
                position = "right side"
            else:
                position = "center"
            
            # determine distance estimate
            if radius < 30:
                distance = "far away"
            elif radius < 80:
                distance = "medium distance"
            else:
                distance = "very close"
            
            return f"Target color '{TARGET_COLOR}' detected on the {position} at X={int(cx)} out of 640. Target is {distance} (radius={int(radius)}px)."
    
    return f"No target detected. Frame is empty."

# Warm up and capture
for _ in range(10):
    cap.read()
    time.sleep(0.05)
ret, frame = cap.read()

if ret:
    observation = build_observation(frame)
    print(f'Observation built:')
    print(f'  {observation}')
    
    # Show the frame
    img_widget = widgets.Image(format='jpeg', width=640, height=480)
    display(img_widget)
    img_widget.value = bgr8_to_jpeg(frame)

---

## YOUR TURN -- Tweak Zone 2: Observation to Decision

Now send that observation to the AI and get a decision back. The full pipeline runs here for the first time -- camera → observation → AI → decision.

Hold different objects in front of the camera and run the cell each time. Does the AI make the right call?

> **Think like an engineer:** The robot is not moving yet -- we are just testing the decision pipeline. Why is it important to test each stage separately before connecting them all together?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THE SYSTEM PROMPT
SYSTEM_PROMPT = """You are the AI brain of a small robot car following a colored target.
Respond with ONLY one word: FORWARD, LEFT, RIGHT, or STOP.
No explanation. No punctuation. Just the single word."""
# ═══════════════════════════════════════

for _ in range(10):
    cap.read()
    time.sleep(0.05)
ret, frame = cap.read()

if ret:
    observation = build_observation(frame)
    print(f'Observation: {observation}')
    
    start = time.time()
    decision = ai_driver.decide(observation, SYSTEM_PROMPT)
    elapsed = time.time() - start
    
    print(f'Decision:    {decision}')
    print(f'Time:        {elapsed:.2f}s')
    print()
    if decision in ['FORWARD', 'LEFT', 'RIGHT', 'STOP']:
        print(f'Valid decision -- robot would: {decision}')
    else:
        print(f'Unexpected response -- robot would STOP for safety')
    
    img_widget2 = widgets.Image(format='jpeg', width=640, height=480)
    display(img_widget2)
    img_widget2.value = bgr8_to_jpeg(frame)

---

## YOUR TURN -- Tweak Zone 3: Improve the Observation

The `build_observation()` function currently only reports on the color target. But the robot also has an **ultrasonic sensor** that can detect obstacles.

Add obstacle information to the observation by tweaking the `SIMULATED_DISTANCE` variable below. This simulates what the ultrasonic sensor would report.

In notebook 09 you will connect the real sensor. For now we simulate it.

> **Think like an engineer:** How should the robot prioritize -- follow the target or avoid the obstacle? What if the obstacle IS the target?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
SIMULATED_DISTANCE = 40   # cm -- simulated ultrasonic reading
                          # try: 10 (very close), 30 (close), 60 (safe), 200 (clear)

SYSTEM_PROMPT_COMBINED = """You are the AI brain of a small robot car.
Your job is to follow a colored target while avoiding obstacles.
If an obstacle is closer than 25cm, STOP or turn to avoid it -- safety first.
Otherwise follow the target.
Respond with ONLY one word: FORWARD, LEFT, RIGHT, or STOP.
No explanation. No punctuation. Just the single word."""
# ═══════════════════════════════════════

def build_observation_with_obstacle(frame, distance_cm):
    """Build observation including both color target and obstacle data."""
    color_obs = build_observation(frame)
    
    if distance_cm < 25:
        obstacle_obs = f"WARNING: Obstacle detected {distance_cm}cm ahead. Path is blocked."
    elif distance_cm < 50:
        obstacle_obs = f"Obstacle detected {distance_cm}cm ahead. Caution recommended."
    else:
        obstacle_obs = f"Path is clear. No obstacles within {distance_cm}cm."
    
    return f"{color_obs} {obstacle_obs}"

for _ in range(10):
    cap.read()
    time.sleep(0.05)
ret, frame = cap.read()

if ret:
    observation = build_observation_with_obstacle(frame, SIMULATED_DISTANCE)
    print(f'Observation: {observation}')
    print()
    
    start = time.time()
    decision = ai_driver.decide(observation, SYSTEM_PROMPT_COMBINED)
    elapsed = time.time() - start
    
    print(f'Decision:    {decision}')
    print(f'Time:        {elapsed:.2f}s')

---

## What Happened?

Think about these questions with your team:

1. Did the AI make the right decision when the simulated obstacle was very close?
2. What happened when there was both a target AND a close obstacle? Which did the AI prioritize?
3. The `build_observation()` function is the bridge between sensor data and AI reasoning. What would happen if it built a wrong observation?
4. Real self-driving cars combine data from cameras, radar, lidar, and GPS. How would you add more sensors to the observation?

---

## CHALLENGE -- Advanced Students

The current observation only describes position (left/center/right) and distance estimate (far/medium/close). 

Add **velocity** to the observation -- is the target moving toward the robot, away from it, or staying still? Compare the current frame's blob position to the previous frame's position to calculate movement direction.

Hint: store the previous blob position in a variable and compare it each frame.

In [ ]:
# YOUR CODE HERE
# ═══════════════════════════════════════
prev_cx = None   # store previous X position here
MOVEMENT_THRESHOLD = 15   # pixels -- minimum movement to count as moving
# ═══════════════════════════════════════

def build_observation_with_velocity(frame):
    global prev_cx
    # detect blob
    # compare to prev_cx
    # add movement description to observation
    # update prev_cx
    pass


---

## Always clean up when you are done!

In [ ]:
motors.cleanup()
cap.release()
print('Motors and camera released.')